In [1]:
import numpy as np;
import time
import scipy.io
import sys
import math

def readInputAsArray(fileName):
    with open(fileName, 'r') as myfile:
        data = myfile.readlines()

    # Strip newline
    for i in range(0, len(data)):
        data[i] = data[i].rstrip()
    # print(data)
    return data

In [2]:
def my_print(name, val):
    print(f'{name}: {val}')

# test=np.array([10])
# my_print('test', test)

In [94]:
def list_str_to_float(arr):
    out = []
    for i in arr:
        out+=[float(i)]
    return out


In [107]:
def append_list_to_np_arr(lst, np_arr):
    if np_arr.shape[0] == 0:
        np_arr = np.array([lst]) 
    else:
        np_arr = np.concatenate((np_arr, [lst]), axis=0) 
    print(np_arr.shape)
    return np_arr

# First make up some inital shape inputs to .mat

In [109]:
# make a 4 sequence fake input to network that is 4 X 147 X 39 
# 2 in enriched and 2 in depleted
def makeSingleShapeArr(seq_file):
    print(">>>>>>>>> In making makeSingleShapeArr ")
    np_all_shape_arr = np.array([])
    
    Seqs = readInputAsArray(seq_file)
    seq_count=1
    for line in Seqs:
        l = line.split()
        if l[1] != '3': # Only consider flanking region of size 3
            continue;
        curr_seq_shape_list = list_str_to_float(l[2:]) # get the shape values only

        if len(curr_seq_shape_list) < 147:
            while len(curr_seq_shape_list) < 147: # if the sequence is shorter than 147
                curr_seq_shape_list+=[0]
            np_all_shape_arr = append_list_to_np_arr(curr_seq_shape_list, np_all_shape_arr)
        else:
            print('>>>>>> Found longer sequences')
            print(len(curr_seq_shape_list))
            start = 0
            while (start < len(curr_seq_shape_list)-146):
                curr_start = start;
                curr_end = 146 + start;
                sub_seq_shape_list = curr_seq_shape_list[curr_start: curr_end+1];
                start+=1
                np_all_shape_arr = append_list_to_np_arr(sub_seq_shape_list, np_all_shape_arr)
        seq_count+=1
    my_print('np_all_shape_arr.shape', np_all_shape_arr.shape)
    return np_all_shape_arr;

# tb_file='/home/yibei/Projects/nucleosome_occupancy/pentamer_table.txt'

###### Now merge the single shape arrs to a whole np array
def concatAllShapeArr(seq_file, All_Shapes):
    np_shape_E = np.array([]) # enriched seqs output

    for i in range(len(All_Shapes)):
        my_print('curr shape is ', All_Shapes[i])
        np_all_shape_arr = makeSingleShapeArr(seq_file + All_Shapes[i] + '.txt')
        if np_shape_E.shape[0] == 0:
            np_shape_E = np_all_shape_arr
        else:
            print(np_shape_E.shape)
            print(np_all_shape_arr.shape)
#             np_shape_E = np.concatenate([np_shape_E, np_all_shape_arr], axis=0)
            np_shape_E = np.stack([np_shape_E, np_all_shape_arr], axis=2)
    return np_shape_E

# seq_file='/home/yibei/Downloads/yibei_predictions/'
#     All_Shapes=['Buckle-FL', 'Buckle', 'EP', 'HelT-FL', 'HelT', 'MGW-FL', 'MGW', 
#                 'Opening-FL', 'Opening', 'ProT-FL', 'ProT', 'Rise-FL', 'Rise', 'Roll-FL',
#                 'Roll', 'Shear-FL', 'Shear', 'Shift-FL', 'Shift', 'Slide-FL', 'Slide', 
#                 'Stagger-FL', 'Stagger', 'Stretch-FL', 'Stretch', 'Tilt-FL', 'Tilt']

seq_file='/Users/yibeijia/Downloads/yibei_predictions/'
np_shape_E = np.array([])
np_shape_D = np.array([])
All_Shapes=['Tilt-FL', 'MGW-FL']
np_shape_E = concatAllShapeArr(seq_file+'test_enriched_', All_Shapes) # depleted seqs output

# All_Shapes=['Roll', 'Stagger']
# np_shape_D = concatAllShapeArr(seq_file+'test_depleted_', All_Shapes) # enriched seqs output

my_print('np_shape_E', np_shape_E.shape) # (NumSeq, 147, 39)
# my_print('np_shape_D', np_shape_D.shape) # (NumSeq, 147, 39)

curr shape is : Tilt-FL
>>>>>>>>> In making makeSingleShapeArr 
(1, 147)
(2, 147)
(3, 147)
(4, 147)
(5, 147)
np_all_shape_arr.shape: (5, 147)
curr shape is : MGW-FL
>>>>>>>>> In making makeSingleShapeArr 
(1, 147)
(2, 147)
(3, 147)
(4, 147)
(5, 147)
np_all_shape_arr.shape: (5, 147)
(5, 147)
(5, 147)
np_shape_E: (5, 147, 2)


## WHY DOES SOME SHPAES NOT HAVE THE CORRECT NUMBER OF BASEPAIRS?

In [86]:
data = {"EnrichedData": np_shape_E, "DepletedData": np_shape_D};
# mat_filename = '/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/sampleSeqs.mat';
mat_filename = '/home/yibei/Projects/data/train_test_data/fakeShapeDNNSeqs.mat'
scipy.io.savemat(mat_filename, data,  do_compression=True);

# Now split data to Train and Test

In [106]:
import math
import pandas as pd
import numpy as np
import scipy.io as sio
from os.path import dirname, join as pjoin
import os.path
import time
import random

tic = time.perf_counter()
Edata_list = np.array([])
Ddata_list = np.array([])
Ndata_list = np.array([])
N=1
print(f"N is: {N}")
# for i in range(1, 51):
for i in range(1, 2):
    print(i)
    # fname = '/scratch2/yibeijia/data/nucleosome_occupancy_' + str(i) + '.mat'
    # fname = '/Users/yibeijia/Downloads/nucleosome_occupancy/data/nucleosome_occupancy_' + str(i) + '.mat'
#     fname = '/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/wormSeqs_'+str(i)+'.mat'
    fname = '/home/yibei/Projects/data/train_test_data/fakeShapeDNNSeqs.mat'
    if os.path.isfile(fname):
        mat_fname = pjoin(fname)
        mat_contents = sio.loadmat(mat_fname)
        
        edata=mat_contents['EnrichedData']
#         ddata=np.array([])# when the seqs are all enriched
        ndata=np.array([])
        ddata=mat_contents['DepletedData']
#         ndata=mat_contents['NeutralData']

        if edata.size >0:
            if Edata_list.size == 0:
                print("Init Elist")
                Edata_list=edata[::N]
            else:
                Edata_list = np.concatenate((Edata_list, edata[::N]), axis=0)
          
        if ddata.size >0:
            if Ddata_list.size == 0:
                print("Init Dlist")
                Ddata_list = ddata[::N];
            else:

                Ddata_list = np.concatenate((Ddata_list, ddata[::N]), axis=0)

        if ndata.size >0:
            if Ndata_list.size == 0:
                print("Init Nlist")
                Ndata_list = ndata[::N];
            else:

                Ndata_list = np.concatenate((Ndata_list, ndata[::N]), axis=0)
    # else:
    #     for j in range(0, 20): See SplitData.py


print("Enriched list size")
print(Edata_list.shape)    
print("Depleted list size")
print(Ddata_list.shape)  
print("Neutral list size")
print(Ndata_list.shape)    
toc=time.perf_counter()
print(f"Create master list took {toc - tic:0.4f} seconds")

D_empty = True
E_empty = True
Ddata_labels=np.array([])
Edata_labels=np.array([])
data_labels=np.array([])
data_list =np.empty((1, 1, 1))
start = True
if Ddata_list.size >0:
    if (Ddata_list.shape[0] > 0):
        print("D non empty")
        D_empty =False
        Ddata_labels = np.zeros(Ddata_list.shape[0])

if Edata_list.size >0:
    if (Edata_list.shape[0] > 0):
        if start:
            # print(Edata_list[0])
            start=False
        print("E non empty")
        E_empty =False
        Edata_labels = np.ones(Edata_list.shape[0])

if (E_empty==False and D_empty==False):
    print("both non empty")
    data_labels =  np.concatenate((Edata_labels, Ddata_labels), axis=0)
    data_list = np.concatenate((Edata_list, Ddata_list), axis=0)
elif (E_empty and D_empty==False):
    print("D non empty, E is empty")
    data_labels =  Ddata_labels
    data_list=Ddata_list
elif (E_empty==False and D_empty):
    print("E non empty, D is empty")
    data_labels = Edata_labels
    data_list=Edata_list

# print(data_list[2][0:4])
## Decide to not mix neutral data in the dataset because training is not with the neutral data.

from sklearn.model_selection import train_test_split
Train_data, Test_data, Train_labels, Test_labels = train_test_split(data_list, data_labels, test_size=0.50, random_state=42)
print(f" All Test data shape is: {Test_data.shape}")
print(f" All Test label shape is: {Test_labels.shape}")
print(f" All Train data shape is: {Train_data.shape}")
print(f" All Train label shape is: {Train_labels.shape}")

Data = {"Test_data" : np.array(Test_data), "Test_labels" : Test_labels}   
sio.savemat('/home/yibei/Projects/data/train_test_data/fakeShapeDNNSeqs_Test.mat',Data, do_compression=True)
print('Wrote test data to mat ')

Data = {"Train_data" : np.array(Train_data), "Train_labels" : Train_labels}   
sio.savemat('/home/yibei/Projects/data/train_test_data/fakeShapeDNNSeqs_Train.mat',Data, do_compression=True)
print('Wrote train data to mat ')

# sio.savemat('/scratch2/yibeijia/data/train_test_data/Test_data.mat',Data, do_compression=True)
# sio.savemat('/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/wormTest.mat', Data
# sio.savemat('/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/wormTest.mat', Data, do_compression=True)
# print(Data)
# sio.savemat('/scratch2/yibeijia/data/train_test_data/Test_data.mat',Data, do_compression=True)


N is: 1
1
Init Elist
Init Dlist
Enriched list size
(2, 147, 39)
Depleted list size
(2, 147, 39)
Neutral list size
(0,)
Create master list took 0.0018 seconds
D non empty
E non empty
both non empty
 All Test data shape is: (2, 147, 39)
 All Test label shape is: (2,)
 All Train data shape is: (2, 147, 39)
 All Train label shape is: (2,)
